In [1]:
import pandas as pd

tracks = pd.read_csv('data/fma_metadata/tracks.csv', index_col=0, header=[0, 1])
print(tracks.shape)
tracks.head()

(106574, 52)


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

In [2]:
tracks['track', 'genre_top'].head()

track_id
2     Hip-Hop
3     Hip-Hop
5     Hip-Hop
10        Pop
20        NaN
Name: (track, genre_top), dtype: str

In [3]:
small = tracks[tracks['set', 'subset'] == 'small']
print("Jumlah track di subset small:", small.shape[0])  # harusnya 8000

Jumlah track di subset small: 8000


In [4]:
genre_labels = small['track', 'genre_top']

# cek apakah ada yang kosong
print("Jumlah label kosong:", genre_labels.isna().sum())

# buang yang kosong (biasanya tidak ada di subset small, tapi tetap dicek)
genre_labels = genre_labels.dropna()
print("Jumlah track dengan label valid:", genre_labels.shape[0])

Jumlah label kosong: 0
Jumlah track dengan label valid: 8000


In [5]:
genre_labels.value_counts()

(track, genre_top)
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

In [6]:
import os

def get_audio_path(track_id, audio_dir='data/fma_small'):
    track_id_str = '{:06d}'.format(track_id)  # jadi 6 digit, misal 2 -> 000002
    return os.path.join(audio_dir, track_id_str[:3], track_id_str + '.mp3')

# tes
print(get_audio_path(2))

data/fma_small\000\000002.mp3


In [7]:
df = pd.DataFrame({
    'track_id': genre_labels.index,
    'genre': genre_labels.values
})
df['filepath'] = df['track_id'].apply(get_audio_path)

df.head()

,track_id,genre,filepath
0,2,Hip-Hop,data/fma_small\000\000002.mp3
1,5,Hip-Hop,data/fma_small\000\000005.mp3
2,10,Pop,data/fma_small\000\000010.mp3
3,140,Folk,data/fma_small\000\000140.mp3
4,141,Folk,data/fma_small\000\000141.mp3


In [8]:
df['file_exists'] = df['filepath'].apply(os.path.exists)

print("File ditemukan:", df['file_exists'].sum())
print("File tidak ditemukan:", (~df['file_exists']).sum())

# lihat contoh yang bermasalah kalau ada
df[~df['file_exists']].head()

File ditemukan: 8000
File tidak ditemukan: 0


,track_id,genre,filepath,file_exists


In [9]:
df['filesize'] = df['filepath'].apply(lambda x: os.path.getsize(x) if os.path.exists(x) else 0)

# file yang mencurigakan (ukurannya sangat kecil, misal < 1KB, biasanya rusak)
suspicious = df[df['filesize'] < 1000]
print("File mencurigakan (kemungkinan corrupt):", suspicious.shape[0])
suspicious

File mencurigakan (kemungkinan corrupt): 0


,track_id,genre,filepath,file_exists,filesize


In [10]:
df.to_csv('data/track_genre_mapping.csv', index=False)
print("Tersimpan:", df.shape)

Tersimpan: (8000, 5)
